In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap

## Prepare two zero pi 

In [2]:
args_all = ut.get_operator_two_zeropi()

In [3]:
[n_theta1, n_theta2, n_theta1_dress, n_theta2_dress, n_theta1_truc, n_theta2_truc,
eval_tot, order_sort, H0, trunc_states, eket0, eket1, eket_tot  ] = args_all

truc = len(trunc_states)
state_tot = [qt.basis(truc, i) for i in range(truc)]

e_ops = [qt.basis(truc, i) * qt.basis(truc, i).dag()
        for i in range(truc)]

# Trans ;  w_ij;    n_theta1; n_theta2; sum
# 00-14 ;  45.925 ;  0.032 ;  0.006 ;  0.038
# 20-14 ;  24.289 ;  0.046 ;  0.001 ;  0.047
logi_space = ['00', '02', '20', '22']
[ w_00_14, w_20_14, n_theta1_00_14, n_theta2_00_14,
n_theta1_20_14, n_theta2_20_14] = ut.get_transition_freq_cnot(args_all)

H_qbt_drive = [ H0, [2*np.pi*n_theta1_truc, ut.drive_gauss_A],
                    [2*np.pi*n_theta1_truc, ut.drive_gauss_B]  ]

n1n1 = True
args = (w_00_14, w_20_14, n_theta1_00_14, n_theta2_00_14,
        n_theta1_20_14, n_theta2_20_14, state_tot, H_qbt_drive, n1n1)


## Get fidelity from drive param

In [4]:
drive_amp_A, drive_amp_B, tg, detune_A, detune_B = [0.07138275557618617,
                0.06263709015005474, 179.5743123263788, -0.23823111049155965, -0.2351910652491388]
# 0.999912   [1.98e-02,  4.64e-02,  4.529436e+02, -3.22e-02, -2.66e-02]
# 0.999937   [1.985e-02,  4.644e-02,  4.5294357e+02, -3.221e-02, -2.657e-02]
pulse_args = {'drive_amp_A': drive_amp_A* (n_theta1_20_14/n_theta1_00_14) ,
            'drive_freq_A': w_00_14 + 2*np.pi*detune_A,
            'drive_amp_B': drive_amp_B ,
            'drive_freq_B': w_20_14 + 2*np.pi*detune_B,
            'gate_time': tg }
tlist = np.linspace(0, tg, num=int(tg))  # total time
prop = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        args=pulse_args,)[-1]  # get the propagator at the final time step
fidelity = ut.cnot_fidelity( prop, state_tot )
print( 'fidelity = ', fidelity, np.log10(1-fidelity))

fidelity =  0.9947686113081093 -2.281383010717746


In [5]:
eket_0 = eket0.todense()
eket_1 = eket1.todense()
idxs = [order_sort.index(i) for i in trunc_states]

jump_op = np.zeros((truc, truc), dtype=complex)
for state in trunc_states[1:]:
    a0i = eket_0[0].conj().T @ eket_0[int(state[0])]
    a0i = eket_0 @ a0i @ eket_0.conj().T
    b0j = eket_1[0].conj().T @ eket_1[int(state[1])]
    b0j = eket_1 @ b0j @ eket_1.conj().T

    eket_truc = np.reshape([eket_tot[i] for i in idxs], (truc, eket1.shape[0]**2))
    ab_00_ij = eket_truc @ np.kron(a0i, b0j) @ eket_truc.conj().T
    jump_op += ab_00_ij

# qt.Qobj(np.reshape(np.round(jump_op, 3), (20,20)))
np.reshape(np.round(jump_op, 2), (20,20))

array([[ 0.  +0.j, -1.  +0.j, -1.  +0.j,  1.  +0.j, -0.49+0.j,  1.27+0.j,
        -0.62+0.j, -0.84+0.j,  0.42+0.j, -1.48+0.j,  0.97+0.j,  0.76+0.j,
         1.2 +0.j, -0.37+0.j, -1.33+0.j,  1.33+0.j, -0.94+0.j,  0.68+0.j,
        -1.24+0.j,  0.9 +0.j],
       [ 0.  +0.j, -0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j,
        -0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j,  0.  +0.j,
         0.  +0.j, -0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j,
        -0.  +0.j,  0.  +0.j],
       [-0.  +0.j,  0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,
         0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j, -0.  +0.j,
        -0.  +0.j,  0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,
         0.  +0.j, -0.  +0.j],
       [-0.  +0.j,  0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,
         0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j, -0.  +0.j,
        -0.  +0.j,  0.  +0.j,  0.  +0.j, -0.  +0.j,  0.  +0.j, -0.  +0.j,
         0.  +0.j, 

In [ ]:
idxs = [order_sort.index(i) for i in trunc_states]
jump_op = np.zeros((truc, truc), dtype=complex)
for state in trunc_states[1:]:
    a0i = eket0[0].getH().dot(eket0[int(state[0])])
    a0i = eket0.dot(a0i).dot(eket0.getH())
    b0j = eket1[0].getH().dot(eket1[int(state[1])])
    b0j = eket1.dot(b0j).dot(eket1.getH())

    eket_truc = ssp.csr_matrix(np.reshape([eket_tot[i] for i in idxs], (truc, eket1.shape[0]**2)))
    ab_00_ij = eket_truc.dot(ssp.kron(a0i, b0j)).dot(eket_truc.getH())
    jump_op += ab_00_ij.toarray()

    # decay_pairs = [
    #     ['00', '02', gamma_2],
    #     ['00', '20', gamma_2],
    #     ['20', '22', gamma_2],
    #     ['02', '22', gamma_2],
    #     ['00', '01', gamma_1],
    #     ['00', '10', gamma_1],
        # ['01', '11', gamma_1],
        # ['10', '11', gamma_1],

        # ['20', '20', gamma_p2],
        # ['02', '02', gamma_p2],
        # ['22', '22', gamma_p2],
        # ['10', '10', gamma_p1],
        # ['01', '01', gamma_p1],
        # ['11', '11', gamma_p1],
            # ]
    # c_op = qt.Qobj(np.zeros((truc, truc)))
    # for i,pair in enumerate(decay_pairs):
    #     c_op+=( np.sqrt(pair[2]) * qt.basis(truc, trunc_states.index(pair[0]))
    #             * qt.basis(truc, trunc_states.index(pair[1])).dag() )

    # jump_op_tphi = qt.qdiags([0] + [1]*(truc - 1), 0) * np.sqrt(gamma_p2)
    

In [ ]:
drive_param_2A = [[0.07557352380680618, 0.045406901210387846, 139.80121562950333, -0.03988304689581973, -0.0463514140425621],
[0.07262631867604984, 0.03998653589305963, 179.84466187159165, -0.03759725943039438, -0.042730473319776696],
[0.06603415954649548, 0.032313257221391674, 215.39818844027002, -0.028110942948351314, -0.03363993397033299],
[0.06561456119771665, 0.03256070030188406, 255.8568219747646, -0.03309632759894371, -0.03658041169233178],
[0.07830073076579484, 0.026863026615025726, 279.9999668380671, -0.031160109465253563, -0.03927095234589164],
[0.07601957910756506, 0.02504258843180761, 309.9999668866966, -0.030272682505848345, -0.03755469639929651],
[0.09150631840114574, 0.022780016816926225, 339.9999145642021, -0.029821737356828013, -0.04530933349733816],
[0.0818997141250141, 0.020776995821780754, 369.9999856598085, -0.026755200445700643, -0.03822141709100123],
[0.06517893233226774, 0.017060124583084493, 399.9999906572392, -0.01611114211193787, -0.025302012374410956],
[0.07368360422310369, 0.01681222219061629, 420.13378612121545, -0.0164934651853555, -0.029706520480420585],
[0.07953654959206298, 0.021779220858655912, 452.90000022645984, -0.009208037275464275, -0.025201661422210253]]

fidelity = []
for drive_param in tqdm(drive_param_2A):
    drive_amp_A, drive_amp_B, tg, detune_A, detune_B = drive_param
    pulse_args = {'drive_amp_A': drive_amp_A* (n_theta1_20_14/n_theta1_00_14) ,
            'drive_freq_A': w_00_14 + 2*np.pi*detune_A,
            'drive_amp_B': drive_amp_B ,
            'drive_freq_B': w_20_14 + 2*np.pi*detune_B,
            'gate_time': tg }
    tlist = np.linspace(0, tg, num=int(tg))  # total time
    prop = qt.propagator( H=H_qbt_drive,
                            t=tlist,
                            args=pulse_args,)[-1]  # get the propagator at the final time step
    fidelity.append(ut.cnot_fidelity( prop, state_tot))
print('fidelity = ',fidelity)

 18%|█▊        | 2/11 [00:06<00:29,  3.24s/it]capi_return is NULL
Call-back cb_f_in_zvode__user__routines failed.
 18%|█▊        | 2/11 [00:12<00:56,  6.29s/it]


KeyboardInterrupt: 

## Plot population transfer

In [ ]:
drive_amp_A, drive_amp_B, tg, detune_A, detune_B = [1.98e-02,  4.64e-02,  4.529436e+02, -3.22e-02, -2.66e-02]
# [0.02145624555393775, 218.48060804705105, -0.005839575487404269],

H_qbt_drive = [ H0, [2*np.pi*n_theta1_truc, ut.drive_gauss_A],
                    [2*np.pi*n_theta1_truc, ut.drive_gauss_B]  ]
pulse_args = {'drive_amp_A': drive_amp_A* (n_theta1_20_14/n_theta1_00_14) ,
            'drive_freq_A': w_00_14 + 2*np.pi*detune_A,
            'drive_amp_B': drive_amp_B ,
            'drive_freq_B': w_20_14 + 2*np.pi*detune_B,
            'gate_time': tg }

tlist = np.linspace(0, tg, num=3*int(tg))  # total time
options = qt.Options(nsteps=10000, store_states=True)
result = {}
logi_state = [0,2]
for state, state_i in enumerate(logi_state):
    for jdx, state_j in tqdm(enumerate(logi_state)):
        state_ij =  str(state_i) + str(state_j)
        state_idx = logi_space.index(state_ij)
        result[state, jdx] = qt.mesolve(
            H=H_qbt_drive,
            rho0=qt.basis( len(trunc_states), state_idx ),
            tlist=tlist,
            e_ops=e_ops,
            args=pulse_args,
            options=options
        )
#####################################################################################
##### Drive and Population Visualization #######
fig, ax = plt.subplots(figsize=(6,4.5), nrows=2, ncols=2)
#ax[0].plot(tlist, drive)
# fig.suptitle('g = %.3f GHz'%g, fontsize=12)
plt.subplots_adjust(wspace=0.3, hspace=0.3)
for state, state_i in enumerate(logi_state[:]):
    for jdx, state_j in enumerate(logi_state[:]):
        for kdx, res in zip(trunc_states, result[state,jdx].expect):
            ax[state][jdx].plot(tlist, res, label="%s"%kdx)
        ax[state][jdx].set_ylim(-0.02,1.02)
        ax[state][jdx].grid()
        ax[state][jdx].set_title('initial state='+str(state_i)+str(state_j))
        # ax[idx][jdx].legend()
    ax[1][state].set_xlabel('Time (ns)')
    ax[state][0].set_ylabel('Population')

handles, labels = ax[1][1].get_legend_handles_labels()
ax[1][1].legend(handles[::-1],
                labels[::-1],
                loc='center left',
                bbox_to_anchor=(1, 1.1)).get_frame().set_alpha(0.1)
pop = np.array(result[0,0].expect[:4])
print('total poulation at t=tg: ', np.sum(pop[:,-1]))

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.plot(tlist, result[0,0].expect[0], label=r"00" )
ax.plot(tlist, result[0,0].expect[2], label=r"20")
ax.plot(tlist, result[0,0].expect[9], label=r"14")
ax.grid()
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Population')
ax.legend()